### What Are Tools in LangChain?

- Tools = Functions that an LLM can use to interact with the outside world

- Normally, an LLM:
  - Only knows text
  - Cannot access real-time data
  - Cannot perform actions

- Why Tools Are Needed?
   - Without tools:
      - User: What's the weather in Kochi?
      - LLM: (guesses or gives outdated info)
   - With tools:
      - LLM → calls Weather API → gets real data → responds correctly
      

-  Calculator Tool
   - Performs math
- Search Tool
   - Queries Google / Wikipedia
- Database Tool
   - Fetches user data
-  API Tool
   - Weather
   - Stock prices
   - Maps

```python
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

```python
from langchain.agents import initialize_agent
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI()

agent = initialize_agent(
    tools=[multiply],
    llm=llm,
    agent="zero-shot-react-description"
)

agent.run("What is 5 times 6?")

```python
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

# Test directly
print(multiply.invoke({"a": 5, "b": 6}))

### What is @tool?

- It’s a decorator that:
  - Wraps a Python function
  - Adds metadata (name, description, input schema)
  - Makes it usable by an LLM agent

- The docstring ("""Multiply two numbers""") becomes the tool description
- Type hints define the input schema

- Internally:
  - LLM sees available tools + descriptions
  - It decides:
    -  “I should call multiply”
  - It generates arguments: { "a": 5, "b": 6 }

___

- In frameworks like LangChain, the LLM is prompted like this:

```raw
You have access to these tools:

1. multiply(a: int, b: int) - Multiply two numbers

If needed, respond in this format:

Action: <tool_name>
Action Input: <json arguments>
```

-  User asks:
   - What is 5 times 6?

- LLM generates text (NOT executing anything):
```raw 
Action: multiply
Action Input: {"a": 5, "b": 6}
```

- This is just plain text output
- LangChain parses this
- Python executes the function
- Result goes back to LLM
   - Observation: 30
- LLM generates final answer
   - Final Answer: The result is 30.


___
___

- 1. Simple Custom Tool (Function-Based)
- 2. Tool with Agent (LLM decides to use it)
- 3. API Tool (Real-World Use)
- 4. Python REPL Tool (Run Code Dynamically)
- 5. Database / Data Tool (Custom Logic)

##### Tool with Agent (LLM decides to use it)

```python
from langchain.tools import tool
from langchain.agents import initialize_agent
from langchain_openai import ChatOpenAI

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = initialize_agent(
    tools=[multiply],
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

agent.run("What is 12 times 7?")

You’ll see:

- Thought
- Action
- Tool call
- Final answer

- agent = initialize_agent(...)
   - This is NOT just an object — it is a loop that combines:
      - LLM (brain)
      - Tools (capabilities)
      - Decision logic (how to act)

- An Agent = LLM + Tools + Reasoning Loop
   - Think (using LLM)
   - Decide (which tool to use)
   - Act (call tool via system)
   - Observe (get result)
   - Repeat until done

```python
agent = initialize_agent(
    tools=[multiply],
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)
```
- LLM (brain)
- Tools (actions)
- Agent Type (VERY IMPORTANT)
   - agent="zero-shot-react-description"
   - This defines HOW the agent thinks
   - "zero-shot-react-description"?
      - This is a reasoning strategy:
      - ReAct = Reason + Act
      - Reason (think) + Act (use tools) + Observe + Repeat

___

#### 3. API Tool (Real-World Use)

```python
import requests
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city"""
    # Example API (replace with real one)
    url = f"https://api.example.com/weather?q={city}"
    
    # Simulated response (since no API key)
    return f"The weather in {city} is 30°C and sunny"

print(get_weather.invoke({"city": "Kochi"}))

____

### 4. Python REPL Tool (Run Code Dynamically)

```python
from langchain.tools import Tool
from langchain_experimental.utilities import PythonREPL

python_repl = PythonREPL()

python_tool = Tool(
    name="python_repl",
    description="Executes Python code",
    func=python_repl.run
)

print(python_tool.run("print(5 * 6)"))

#### 5. Database / Data Tool (Custom Logic)

```python
from langchain.tools import tool

fake_db = {
    "user_1": "Ashik",
    "user_2": "Rahul"
}

@tool
def get_user(user_id: str) -> str:
    """Fetch user name from database"""
    return fake_db.get(user_id, "User not found")

print(get_user.invoke({"user_id": "user_1"}))

#### 6. Multi-Tool Agent (Real Power)

- Combine multiple tools

```python
from langchain.tools import tool
from langchain.agents import initialize_agent
from langchain_openai import ChatOpenAI

@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = initialize_agent(
    tools=[add, multiply],
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

agent.run("What is (5 + 3) * 2?")
```


### 7. Class-Based Tool (Advanced)

```python
from langchain.tools import BaseTool

class MultiplyTool(BaseTool):
    name = "multiply"
    description = "Multiply two numbers"

    def _run(self, a: int, b: int):
        return a * b

    async def _arun(self, a: int, b: int):
        raise NotImplementedError()

tool = MultiplyTool()
print(tool.run({"a": 4, "b": 5}))

### 8. Tool with Structured Input (Better Control)

```python
from pydantic import BaseModel
from langchain.tools import StructuredTool

class MultiplyInput(BaseModel):
    a: int
    b: int

def multiply_func(a: int, b: int) -> int:
    return a * b

multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

print(multiply_tool.invoke({"a": 3, "b": 9}))

### 9. Real-World Style Tool (Your Domain – Kafka)

```python
from langchain.tools import tool

@tool
def get_kafka_logs() -> str:
    """Fetch recent error logs from Kafka"""
    # Simulated logs
    return "Error: Partition failure at node 3"

@tool
def analyze_logs(log: str) -> str:
    """Analyze logs and find issue"""
    if "failure" in log:
        return "System failure detected in partition"
    return "No issue found"